In [1]:
# Initialize graph

import geopandas as gpd
import networkx as nx
from contextlib import contextmanager
import time
import osmnx as ox
import numpy as np

@contextmanager
def perfTimer(action=""):
    start = time.perf_counter()
    yield
    end = time.perf_counter()
    print(f"Elapsed time to {action}: {round(end - start, 2)}s")

with perfTimer("read nodes and edges of mabalacat+angeles City"):
    nodes = gpd.read_file("graph/transit_graph/walking_graph/mabalacatAngelesWalking_nodes.geojson")
    edges = gpd.read_file("graph/transit_graph/walking_graph/mabalacatAngelesWalking_edges.geojson")

with perfTimer("Fix index of nodes and edges"):
    nodes = nodes.set_index("osmid")
    edges = edges.set_index(["u", "v", "key"])


with perfTimer("convert np arrays to lists"):
    for attr in edges.columns:
        new_values = []

        for val in edges[attr]:
            if isinstance(val, np.ndarray):
                new_values.append(val.tolist())
            else:
                new_values.append(val)

        edges[attr] = new_values

with perfTimer("initialize MultiDiGraph"):
    G = ox.graph_from_gdfs(nodes, edges)

c:\Users\Adriane\Documents\bscs practice 2\sakenph-backend\venv\Lib\site-packages\geopandas\io\file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


Elapsed time to read nodes and edges of mabalacat+angeles City: 4.56s
Elapsed time to Fix index of nodes and edges: 0.02s
Elapsed time to convert np arrays to lists: 0.51s
Elapsed time to initialize MultiDiGraph: 1.03s


In [ ]:
# Test Yen's

import osmnx as ox
from contextlib import contextmanager
import time

@contextmanager
def perfTimer(action=""):
    start = time.perf_counter()
    yield
    end = time.perf_counter()
    print(f"Elapsed time to {action}: {round(end - start, 2)}s")

start = 12093547 # Astro park beside purple jeep terminal
end = 9191058305 # Hau beside bart mall

with perfTimer("do Yen's"):
    pathList = ox.routing.k_shortest_paths(G, start, end, 3)

for i in pathList:
    print(i)

Elapsed time to do Yen's: 0.0s
[12093547, 317179340, 675646119, 7729495329, 675641141, 9570056689, 1128117764, 322054705, 3341100920, 3341100875, 3388973403, 3388973405, 3388973406, 3388973407, 3341100892, 317176865, 8992824705, 457888678, 9196110871, 7675063407, 6846711041, 9196090037, 12035336006, 12035336005, 449659809, 9196090029, 12595304824, 449659810, 11258891594, 11258891598, 13692022598, 9196021615, 527827933, 10101099008, 9196021616, 7675100203, 7675695295, 527827924, 854339638, 13479138054, 13479138061, 7675063465, 7844953666, 10095114316, 9196021556, 854338664, 449658386, 9196021573, 9196021557, 6118354752, 449658387, 311718967, 9196000168, 7676313057, 9196000167, 313908262, 7676077658, 3343727920, 3343727918, 3343727917, 11269451251, 313908392, 313910357, 438033707, 7713520922, 7693476941, 313909113, 438033711, 457306235, 3339627301, 7675735513, 3339627332, 3339627335, 3339627323, 9183498724, 3339627340, 312291686, 7675733173, 8856524323, 518261998, 13511942695, 6828588389

In [ ]:
# IM CHECKING IF THERE ARE ANY SAME U V PAIRS
import json
dict = {'13': 23, '13': 24}

theDict = {}
sameKeyPair = {}

with open('graph/originalGraph/mabalacatAngeles_edges.geojson', 'r') as file:
    jsonData = json.load(file)


for i in jsonData['features']:
    u = i['properties']['u']
    v = i['properties']['v']

    if u in theDict.keys():
        if v in theDict[u]:
            sameKeyPair[u] = v
        else:
            theDict[u].append(v)
    else:
        theDict[u] = []
        theDict[u].append(v)

# Result: yes there is.

In [ ]:
# Load mabalacatAngeles_{nodes & edges}.geojson (no longer needed maybe)

with open('graph/originalGraph/mabalacatAngeles_nodes.geojson', 'r') as file:
    nodes_jsonData = json.load(file)

with open('graph/originalGraph/mabalacatAngeles_edges.geojson', 'r') as file:
    edges_jsonData = json.load(file)

In [ ]:
import copy
import json
import os

nodeIdIndex = 0
edgeIdIndex = 0

def addJeepRoute(
        nodeList: list[int], 
        routeName: str
    ):
    '''
    Creates a node and edges file in GeoJSON format from a list of nodes that constitutes a path aligned to the jeepney route

    Throws an error if an osmid from the node list does not exist in the graph
    Throws an error if two adjacent nodes in the list do not have an edge connecting one another in the graph

    
    Parameters
    ----------
    nodeList
        It should be a list of nodes that constitutes a path aligned to the jeepney route.
        It will throw an error if the node_id in the graph does not exist, or if two adjacent
        nodes in the list dont have an edge connecting each other

    routeName
        Name of the jeepney route, preferably what the label on the side of the jeepney says

    Returns
    ----------
    None
    '''
    global nodeIdIndex; nodeIdIndex += 1
    global edgeIdIndex; nodeIdIndex += 1
    osmIdDict = {}

    # Tracks the ids of walking nodes' equivalent mode nodes


    # Reads the nodes, edges json of the original graph
    with open('graph/original_graph/mabalacatAngeles_nodes.geojson', 'r') as file:
        nodes_jsonData = json.load(file)
    with open('graph/original_graph/mabalacatAngeles_edges.geojson', 'r') as file:
        edges_jsonData = json.load(file)

    # Json file of the jeepney route nodes to write
    jeepRouteNodesJson = {
        "type": "FeatureCollection",
        "name": f"{routeName}_nodes",
        "crs": {
            "type": "name", 
            "properties": { 
                "name": "urn:ogc:def:crs:OGC:1.3:CRS84" 
            } 
        },
        "features": [

        ]
    }

    # Json file of the jeepney route edges to write
    jeepRouteEdgesJson = {
        "type": "FeatureCollection",
        "name": f"{routeName}_edges",
        "crs": {
            "type": "name", 
            "properties": { 
                "name": "urn:ogc:def:crs:OGC:1.3:CRS84" 
            } 
        },
        "features": [

        ]
    }

    
    # New osm_id formula (a * {corresponding osmid of walking node} + b)
    a = 105002; b = 67203817 
    createNewOsmId = lambda ogId: a * ogId + b
    

    nodesVisited = {}

    # --------------------- FIRST NODE -------------------------#
    # Looks for the first node id from the route list 
    # inside the graph and gets data about its original node

    jeepNodeToAdd = None
    for i in nodes_jsonData['features']:
        if i['properties']['osmid'] == nodeList[0]:
            jeepNodeToAdd = copy.deepcopy(i)

            # Creates a unique osm_id for the node of the route
            # by this formula (105002 * node_id + 67203817)
            # newOsmId = createNewOsmId(nodeList[0])

            # Assigns id to new mode node
            newOsmId = nodeIdIndex
            osmIdDict[nodeList[0]] = newOsmId

            # Changes old osm_id to new one
            jeepNodeToAdd['properties']['osmid'] = newOsmId

            jeepNodeToAdd['properties']['mode'] = {
                "type": "jeep",
                "details": {
                    "name": routeName
                }
            }

            # Makes sure that newly added node is not duplicated in
            # case it is traversed again in the route
            nodesVisited[nodeList[0]] = copy.deepcopy(jeepNodeToAdd)
            break

    if (jeepNodeToAdd):
        jeepRouteNodesJson["features"].append(jeepNodeToAdd)
    else:
        raise KeyError(f"Node with id \"{nodeList[0]}\" is not in the graph.")

    # Outbound and inbound jeep node weights
    # 470 for inbound weight is designated to add penalty for boarding,
    # used to discourage the algorithm from picking short jeepney trips
    outJeepWeight = 10; inJeepWeight = 200

    # Function that creates a new feature for the edgesJson['features'] dictionary
    inOut_edgeFeature = lambda u, v, id, nodeFeatures, direction: {
        "type": "Feature",
        "properties": {
            "u": u,
            "v": v,
            "key": 0,
            "osmid": str(id),
            "length": 0.01,
            "weight": inJeepWeight if direction=='in' else outJeepWeight if direction=='out' else '0'
        },
        "geometry": {
            "type": "LineString",
            "coordinates": [
                [nodeFeatures['properties']['x'], nodeFeatures['properties']['y']],
                [nodeFeatures['properties']['x'], nodeFeatures['properties']['y']]
            ]
        }
    }

    # Add inbound jeep edge
    jeepRouteEdgesJson['features'].append(inOut_edgeFeature(nodeList[0], newOsmId, edgeIdIndex, jeepNodeToAdd, 'in'))

    # Add outbound jeep edge
    jeepRouteEdgesJson['features'].append(inOut_edgeFeature(newOsmId, nodeList[0], edgeIdIndex, jeepNodeToAdd, 'out'))

    # ---------------- SECOND TO THE LAST NODE -------------------#
    for m in range(1, len(nodeList)):
        nodeIdIndex += 1; edgeIdIndex += 1
        jeepNodeToAdd = None

        # Check if node has not been visited already in the route
        if nodeList[m] not in nodesVisited.keys():

            # Find node in the graph
            for i in nodes_jsonData['features']:

                # Check if node matches
                if i['properties']['osmid'] == nodeList[m]:

                    # Get node feature details
                    jeepNodeToAdd = copy.deepcopy(i)

                    newOsmId = nodeIdIndex
                    osmIdDict[nodeList[m]] = newOsmId

                    jeepNodeToAdd['properties']['osmid'] = newOsmId

                    # Add walking node and its features to visited list
                    nodesVisited[nodeList[m]] = copy.deepcopy(jeepNodeToAdd)
                    break

            if (jeepNodeToAdd):
                jeepNodeToAdd["properties"]["mode"] = {
                    "type": "jeep",
                    "details": {
                        "name": routeName
                    }
                }
                jeepRouteNodesJson["features"].append(jeepNodeToAdd)
            else:
                raise KeyError(f"Node with id \"{nodeList[m]}\" is not in the graph.")

            # Inbound jeep edge
            jeepRouteEdgesJson['features'].append(inOut_edgeFeature(nodeList[m], newOsmId, edgeIdIndex, jeepNodeToAdd, 'in'))

            # Outbound jeep edge
            jeepRouteEdgesJson['features'].append(inOut_edgeFeature(newOsmId, nodeList[m], edgeIdIndex, jeepNodeToAdd, 'out'))

        else:
            newOsmId = osmIdDict[nodeList[m]]

        edgeDetails = None

        for i in edges_jsonData['features']:
            if i['properties']['u'] == nodeList[m-1] and i['properties']['v'] == nodeList[m] and i['properties']['key'] == 0:
                edgeDetails = copy.deepcopy(i)
                break
        
        jeepSpeed = 3.89 # m/s
        if (edgeDetails):
            geometry = edgeDetails['geometry']
            length = edgeDetails['properties']['length']
            edgeIdIndex += 1

            # Calculated from length by speed which corresponds to how long
            # it takes to traverse an edge based on jeepney's average speed
            # in seconds
            weight = round(round(length, 2) / jeepSpeed, 2)

            jeepRouteEdgesJson['features'].append({
                "type": "Feature",
                "properties": {
                    "u": osmIdDict[nodeList[m-1]],
                    "v": newOsmId,
                    "key": 0,
                    "osmid": str(edgeIdIndex),
                    "length": length,
                    "weight": weight
                },
                "geometry": geometry
            })
        else:
            warningInput = input(
                f"Edge with u({nodeList[m-1]}) and v({nodeList[m]}) pair does not exist!\n" \
                "Are you sure you want to continue? Route may have disconnected paths.\n" \
                "Input \"yes\" to continue with exporting: "
            )

            if warningInput.lower() != 'yes':
                raise KeyError('Edge with corresponding (u,v) pair does not exist in the graph.')
            
    if not os.path.exists(f"graph/transit_graph/jeep_routes/{routeName}"):
        os.makedirs(f"graph/transit_graph/jeep_routes/{routeName}")

    with open(f'graph/transit_graph/jeep_routes/{routeName}/{routeName}_nodes.geojson', 'w') as file:
        json.dump(jeepRouteNodesJson, file, indent=1)

    with open(f'graph/transit_graph/jeep_routes/{routeName}/{routeName}_edges.geojson', 'w') as file:
        json.dump(jeepRouteEdgesJson, file, indent=1)

# trike = 6.48m/s
# jeep = 3.89m/s
# walk = 1.25m/s



purpleJeepList = [
    3289714296, 3289714287, 3289714288, 3289714289, 5548939409, 2858136284, 12154752, 9174981631,
    7920874965, 9174981623, 9174978808, 9174978810, 854298207, 508035457, 12022633769, 12022633771,
    1843464661, 6471612671, 12272206774, 12272206781, 518296295, 7675114210, 7675097457, 8856752776,
    660578739, 1843464640, 660578733, 660578714, 283119889, 6266014252, 9175071247, 6266014256,
    6266014261, 13478908834, 314892198, 3192760788, 7690166269, 7170588670, 9456891490, 9456891488,
    3343162812, 3350956127, 3377833798, 7690178685, 3377833806, 3377833811, 3339677203, 7690166237,
    9176768661, 538127247, 9176771003, 9176770995, 1853859118, 7675733172, 6869893309, 311674612,
    13476114367, 9176915517, 1301475778, 9176915518, 6869976572, 279718156, 279718155, 529856382,
    4952801798, 457281472, 517417851, 279718148, 3556332238, 6831097023, 3393291352, 7676166990,
    1872304937, 9191121906, 529853828, 5549726320, 7675801284, 8318725917, 854202919, 7675801282,
    6257635296, 2627302000, 10979944928, 7676009004, 13476558708, 2627302003, 2627302029, 854186888,
    9191058308, 12206389590, 2892712059, 12206389589, 2627301958, 9191058305, 1841740882, 514546844,
    846548468, 8304931722, 312297107, 312297509, 9191021570, 9191021573, 6828588336, 312294586, 13509227792,
    6870297706, 6355364996, 9183623297, 13509227798, 8302829218, 13509227786, 13509227795, 9183623311, 13509227789, 311674987, 312291693, 313910490,
    10113743412, 6828588389, 13511942695, 518261998, 8856524323, 7675733173, 312291686, 3339627326, 3339627327,
    3339627338, 3378951301, 7675735513, 3339627332, 312290705, 9183484508, 9183484509, 312290206,
    312290039, 311685317, 9183433986, 311675085, 3461062895, 1853898866, 457304698, 457304699,
    457304700, 457304687, 457304701, 313909603, 457304686, 313909477, 457304684, 312286063,
    10981690263, 457304685, 311685631, 311685572, 3339675390, 13595164001, 13595164003, 1853859132, 371318628, 371318598,
    6261997074, 6261997071, 9176751278, 371319548, 7690166226, 7690166243, 9176713770, 9176713731,
    311674860, 9839968425, 10761230866, 10761230868, 9839968433, 3339677215, 7141979357, 7141979356,
    7690166266, 7141979338, 10673981164, 8180600311, 13478908836, 3192760788, 314892198, 13478908834,
    6266014261, 6266014256, 9175071247, 6266014252, 283119889, 660578714, 660578733, 1843464640,
    660578739, 8856752776, 7675097457, 7675114210, 518296295, 12272206781, 12272206774, 6471612671,
    1843464661, 12022625863, 12022633772, 283119899, 9174978813, 9174978804, 9174978797, 7920872766, 485657698,
    5548939411, 3289714301, 3289714306, 1540323475, 3289714286, 3289714296, 1540323472, 3289717496,
    1540323469, 5548940641, 6814957201, 507044703, 8333736630, 1540323460, 10897890037, 1540323459,
    10897890066, 12696282413, 1540323454, 502842658, 1540323435, 11222235956, 1540323431, 11222235964
]

MabalacatDauAngeles = [
    1164921469, 338870504, 1746305271, 9281253746,
    6721960062, 9281253739, 10915569984, 1676875041, 10115868925,
    27170580, 1127578337, 1127578280, 1127578261, 9873662651,
    10089451380, 702805363, 373053399, 373053562, 7672709922,
    11953603894, 11953603901, 373050429, 6754430756, 373055864,
    9565026086, 9565026083, 10915569993, 10094865002, 373058167,
    664675389, 13523873250, 13523873242, 373057767, 373059487,
    6754527012, 619711918, 4583729567, 27170582, 373060607,
    1746305070, 9200214420, 9200214427, 6128693193, 6128693190,
    7675886492, 6128693197, 9200214479, 9200214470, 6128693194,
    6755439378, 9200214493, 9200214484, 9200214495, 8995666502,
    8995666494, 13508535492, 8584482239, 13508535486, 8584482242,
    373067218, 373066911, 9902614009, 10915511848, 10096730144,
    7675841321, 373066663, 660080446, 4417021930, 373072324,
    373073903, 660080430, 9200023907, 10915476579, 373071791,
    10100121266, 10915476592, 9200023909, 373157342, 6754485678,
    9196209156, 13511838165, 9196209149, 8855984724, 13512727130,
    1287153547, 373074467, 9914956813, 9188090059, 9188090051,
    10100109626, 10915479710, 4417018186, 10915479713, 667984263,
    9188090067, 13512505319, 10915479715, 9188090072, 660080329,
    8277757188, 9187885413, 8277757197, 6789418210, 7697805105,
    10911198636, 7697805113, 456858339, 9187898741, 10911198632,
    9187898743, 9187898752, 5431180046, 8276793868, 1355844593,
    8276793869, 12758814586, 7045813348, 7697797625, 5383866267,
    1717305193, 7294252054, 5383866270, 7294252048, 6265418912,
    6265418908, 7294252050, 6789112453, 1355844557, 660114998,
    522303093, 4588450120, 9169978795, 660115002, 9169978796,
    660114999, 9169978790, 317278751, 522302845, 9169978789,
    13164687872, 13164687877, 7678595739, 7678595730, 660114982,
    522302774, 8277920070, 9170072420, 9170056510, 9170056511,
    9170056497, 9170056488, 522302671, 9170056487, 317288869,
    9170056483, 9187628054, 8278178678, 13250223434, 7674980613,
    10111017985, 9187628055, 7357508289, 372577036, 522301886,
    372578357, 7357508288, 372546258, 538858115, 522298465,
    7675045550, 522298220, 10911031191, 522298412, 11084719234,
    11084719235, 283119886, 6266037530, 372443621, 360697784,
    372528399, 6732277546, 11070696329, 9187839297, 6732277551,
    6732277555, 6732277553, 11138324288, 6732277533, 11138333012,
    6732277535, 11138333010, 7673464378, 8682825680, 2748923904,
    8682825683, 8682825685, 13504279964, 7357315008, 13504279973,
    7357315004, 7357315018, 7357332269, 315735804, 508035441,
    314854754, 7695561764, 518296269, 6814692361, 3289714300,
    3289714306, 1540323475, 3289714286
]

greyJeep = [ 
    518296277, 6405400943, 1223928098, 1540323441, 518296275, 677655450, 3343747698, 2858136284, 12154752, 9174981631, 
    7920874965, 9174981623, 9174978808, 9174978810, 854298207, 508035457, 12022633769, 12022633771, 1843464661, 6471612671, 
    12272206774, 12272206781, 518296295, 7675114210, 7675097457, 8856752776, 660578739, 1843464640, 660578733, 660578714, 
    283119889, 6266014252, 9175071247, 6266014256, 6266014261, 13478908834, 314892198, 3192760788, 7690166269, 7170588670, 
    9456891490, 9456891488, 3343162812, 3350956127, 3377833798, 7690178685, 3377833806, 3377833811, 3339677203, 7690166237,
    9176768661, 538127247, 9176771003, 9176770995, 1853859118, 7675733172, 6869893309, 311674612, 13476114367, 9176915517,
    1301475778, 9176915518, 6869976572, 279718156, 279718155, 529856382, 4952801798, 457281472, 517417851, 279718148,
    3556332238, 6831097023, 3393291352, 7676166990, 1872304937, 9191121906, 529853828, 5549726320, 312296969, 7675835589, 
    3393291362, 1872304943, 9191109020, 3393291365, 6831097024, 3556332238, 279718148, 517417851, 457281472, 4952801798, 
    529856382, 279718155, 279718156, 6869976572, 9176915518, 1301475778, 9176915517, 13476114367, 311674612, 6869893309, 
    7675733172, 1853859118, 9176770995, 9176771003, 538127247, 9176768661, 7690166237, 3339677203, 538127234, 3377833803, 
    3377833810, 3377833805, 3377833800, 9839968433, 3339677215, 7141979357, 7141979356, 7690166266, 7141979338, 10673981164, 
    8180600311, 13478908836, 3192760788, 314892198, 13478908834, 6266014261, 6266014256, 9175071247, 6266014252, 283119889, 
    660578714, 660578733, 1843464640, 660578739, 8856752776, 7675097457, 7675114210, 518296295, 12272206781, 12272206774,
    6471612671, 1843464661, 12022625863, 12022633772, 283119899, 9174978813, 9174978804, 9174978797, 7920872766, 485657698,
    5548939411, 3289714301, 3289714306, 1540323475, 3289714286, 3289714296, 1540323472, 3289717496,
    1540323469, 5548940641, 6814957201, 507044703, 8333736630, 1540323460, 10897890037, 1540323459, 10981724689, 1540323467,
    8693517510, 7920872738, 10298112466, 10298120821
]

greepJeep = [
    538127234, 371320824, 9176713742, 311674860, 9176713731,
    9176713770, 7690166243, 7690166226, 371319548, 9176751278,
    6261997071, 6261997074, 371318598, 371318628, 1853859132,
    13595164003, 13595164001, 3339675390, 311685572, 311685631,
    457304685, 10981690263, 312286063, 457304684, 313909477,
    457304686, 313909603, 7198601340, 9461721535, 7198611426,
    9460738254, 9460738264, 313909934, 7198611429, 311675085,
    9183433986, 311685317, 312290039, 312290206, 9183484509,
    9183484508, 312290705, 3339627335, 3339627323, 9183498724,
    3339627340, 312291686, 7675733173, 8856524323, 518261998,
    13511942695, 6828588389, 10113743412, 313910490, 312291693, 311674987,
    13509227789, 9183623311, 13509227795, 13509227786, 8302829218,
    13509227798, 9183623297, 6355364996, 6870297706, 13509227792,
    312294586, 6828588336, 9191021573, 9191021570, 312297509,
    312297107, 11098875724, 846547074, 12206389584, 854186888,
    2627302029, 2627302003, 13476558708, 7676009004, 10979944928,
    2627302000, 6257635296, 7675801282, 854202919, 8318725917,
    7675801284, 5549726320, 312296969, 7675835589, 3393291362,
    1872304943, 9191109020, 3393291365, 516063451, 311674704,
    311674706, 311674707, 311674708, 13478908825, 311674709,
    516063467, 516063463, 516063460, 876562484, 6869941657,
    1301475776, 13476114367, 311674612, 6869893309, 7675733172,
    1853859118, 9176770995, 9176771003, 538127247, 9176768661,
    7690166237, 3339677203
]

MabalacatDauAngelesNorth = [
    11468813213, 518296269, 7695561764, 314854754, 508035441, 315735804, 7357332269, 7357315018, 7357315004, 13504279973, 7357315008, 13504279964, 8682825685, 8682825683,
    2748923904, 8682825681, 1842375233, 6732277534, 8682963010, 7675047513, 6727297893, 11138324289, 9188048889, 9188048896, 315737081, 8682986717, 8682963011, 6732280689,
    1703686178, 12093573, 6727297891, 6732133037, 9187839296, 6732277546, 372528399, 360697784, 372443621, 6266037530, 283119886, 11084719235, 11084719234, 522298412,
    10911031191, 522298220, 7675045550, 522298465, 538858115, 372546258, 7357508288, 372578357, 522301886, 372577036, 7357508289, 9187628055, 10111017985, 7674980613,
    13250223434, 8278178678, 9187628054, 10911041326, 10911041327, 10911041325, 10911041330, 9170056487, 522302671, 9170056488, 9170056497, 9170056511, 9170056510,
    9170072420, 8277920070, 522302774, 660114982, 7678595730, 7678595739, 13164687877, 13164687872, 9169978789, 522302845, 317278751, 9169978790, 660114999, 9169978796, 660115002, 
    9169978795, 4588450120, 522303093, 660114998, 1355844557, 6789112453, 7294252050, 6265418908, 6265418912, 7294252048, 5383866270, 7294252054, 1717305193,
    5383866267, 7697797625, 7045813348, 12758814586, 8276793869, 1355844593, 8276793868, 5431180046, 9187898752, 9187898743, 10911198632, 10911198627, 10911198629,
    10911198633, 10911198634, 7697805105, 6789418210, 8277757197, 9187885413, 8277757188, 660080329, 9188090072, 10915479715, 13512505319, 9188090067, 667984263, 10915479713, 
    4417018186, 10915479710, 10100109626, 9188090051, 9188090059, 9914956813, 373074467, 1287153547, 13512727130, 8855984724, 9196209149, 13511838165, 9196209156, 6754485678, 
    373157342, 9200023909, 10915476592, 10100121266, 373071791, 10915476579, 9200023907, 660080430, 373073903, 373072324, 4417021930, 660080446, 373066663, 7675841321, 
    10096730144, 10915511848, 10915511851, 10915511854, 373066911, 373067218, 8584482242, 13508535486, 8584482239, 13508535492, 8995666494, 8995666502, 9200214495, 9200214484, 
    9200214493, 6755439378, 6128693194, 9200214470, 9200214479, 6128693197, 7675886492, 6128693190, 6128693193, 9200214427, 9200214420, 1746305070, 373060607, 27170582, 
    4583729567, 619711918, 6754527012, 373059487, 373057767, 13523873242, 13523873250, 664675389, 373058167, 10094865002, 10915569993, 9565026083, 9565026086, 373055864, 
    6754430756, 373050429, 11953603901, 11953603894, 7672709922, 373053562, 373053399, 702805363, 10089451380, 9873662651, 1127578261, 1127578280, 1127578337, 27170580, 
    1127578267, 1676875041, 10915569984, 9281253739, 6721960062, 9281253746, 1746305271, 338870504, 1746305338
]



addJeepRoute(purpleJeepList, "Checkpoint-HolyAngelUniversity-Highway-Angeles")
addJeepRoute(MabalacatDauAngeles, "Mabalacat-Dau-Angeles-Southbound")
addJeepRoute(MabalacatDauAngelesNorth, "Mabalacat-Dau-Angeles-Northbound")
addJeepRoute(greyJeep, 'Checkpoint-HolyAngelUniversity')
addJeepRoute(greepJeep, 'Marisol-Pampang')


In [ ]:
# Checking node data in G

print(G.nodes[345426646767391])
print(G.nodes[3289714287])

{'osmid': 345426646767391, 'y': 15.1691696, 'x': 120.5880267, 'street_count': 3, 'highway': None, 'ref': None}


{'y': np.float64(15.1691696),
 'x': np.float64(120.5880267),
 'street_count': np.int32(3)}

In [ ]:
# Checking edge data in G
u = 345426646767391
v = 3289714287

print(G.get_edge_data(u, v))

{0: {'weight': 10}}


In [ ]:
# Reverse Engineer function (Used to find walking edge)

def reverseEngineer(number):
    return int((number - 67203817) / 105002)

jeepNode = 350672662504597
print(reverseEngineer(jeepNode))

3339675390


In [ ]:
# Add weight for walking edges

jsonToWrite = {
    "type": "FeatureCollection",
    "name": "mabalacatAngelesGraph_edges",
    "crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:OGC:1.3:CRS84" } },
    "features": [
    ]
}

with open('graph/original_graph/mabalacatAngeles_edges.geojson', 'r') as file:
    jsonData = json.load(file)


# m/s
walkingSpeed = 1.25

for i in jsonData['features']:
    i['properties']['weight'] = round(round(i['properties']['length'], 2) / walkingSpeed, 2)


    jsonToWrite['features'].append(i)


with open('graph/transit_graph/walking_graph/mabalacatAngelesWalking_edges.geojson', 'w') as file:
    json.dump(jsonToWrite, file)





In [109]:
# Add mode for walking nodes

jsonToWrite = {
    "type": "FeatureCollection",
    "name": "mabalacatAngelesGraph_nodes",
    "crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:OGC:1.3:CRS84" } },
    "features": [
    ]
}

with open('graph/original_graph/mabalacatAngeles_nodes.geojson', 'r') as file:
    jsonData = json.load(file)

for i in jsonData['features']:
    i['properties']['mode'] = {
        "type": "walk"
    }

    jsonToWrite['features'].append(i)

with open('graph/transit_graph/walking_graph/mabalacatAngelesWalking_nodes.geojson', 'w') as file:
    json.dump(jsonToWrite, file)


In [ ]:
# Export graph to nodes and edges geojson files using G
import osmnx as ox

nodes, edges = ox.graph_to_gdfs(G)
edges.to_file("graph/transit_graph/jeep_routes/mabalacatAngelesJeep_edges.geojson", driver='GeoJSON')

In [18]:
# Add jeepney route JSONS to the graph (Requires G to be initialized)

import json
from shapely.geometry import shape

with open("graph/transit_graph/jeep_routes/Checkpoint-HolyAngelUniversity-Highway-Angeles/Checkpoint-HolyAngelUniversity-Highway-Angeles_edges.geojson") as f:
    purpleEdges = json.load(f)

with open("graph/transit_graph/jeep_routes/Checkpoint-HolyAngelUniversity-Highway-Angeles/Checkpoint-HolyAngelUniversity-Highway-Angeles_nodes.geojson") as f:
    purpleNodes = json.load(f)


for feature in purpleNodes["features"]:
    props = feature["properties"].copy()
    geom = feature["geometry"]
    node_id = props["osmid"]
    lon, lat = geom["coordinates"]
    props["x"] = lon
    props["y"] = lat


    G.add_node(node_id, **props
    )


for feature in purpleEdges["features"]:
    props = feature["properties"]
    geom = feature["geometry"]

    u = props.pop("u")
    v = props.pop("v")

    G.add_edge(
        u,
        v,
        geometry=shape(geom),   # keep full geometry
        **props
    )

In [3]:
# Loops through every route json

import os
import json
from shapely.geometry import shape

relativePath = "graph/transit_graph/jeep_routes"

for route in os.listdir(relativePath):

    for file in os.listdir(f"{relativePath}/{route}"):

        if file.endswith('_nodes.geojson'):
            with open(f"{relativePath}/{route}/{file}") as f:
                nodesJson = json.load(f)

            for feature in nodesJson["features"]:
                props = feature["properties"].copy()
                geom = feature["geometry"]
                node_id = props["osmid"]
                lon, lat = geom["coordinates"]
                props["x"] = lon
                props["y"] = lat

                G.add_node(node_id, **props)

        elif file.endswith('_edges.geojson'):
            with open(f"{relativePath}/{route}/{file}") as f:
                edgesJson = json.load(f)

            for feature in edgesJson["features"]:
                props = feature["properties"]
                geom = feature["geometry"]

                u = props.pop("u")
                v = props.pop("v")

                G.add_edge(
                    u,
                    v,
                    geometry=shape(geom),
                    **props
                )
        else:
            print("NOTHING!")


In [2]:
# Loops through every route json 2
import os
import json
from shapely.geometry import shape

relativePath = "graph/transit_graph/jeep_routes"

for route in os.listdir(relativePath):

    for file in os.listdir(f"{relativePath}/{route}"):

        if file.endswith('_nodes.geojson'):
            with open(f"{relativePath}/{route}/{file}") as f:
                nodesJson = json.load(f)

            for feature in nodesJson["features"]:
                props = feature["properties"].copy()
                geom = feature["geometry"]
                node_id = props["osmid"]
                lon, lat = geom["coordinates"]
                props["x"] = lon
                props["y"] = lat

                G.add_node(node_id, **props)

        elif file.endswith('_edges.geojson'):
            with open(f"{relativePath}/{route}/{file}") as f:
                edgesJson = json.load(f)

            for feature in edgesJson["features"]:
                props = feature["properties"]
                geom = feature["geometry"]

                u = props.pop("u")
                v = props.pop("v")

                G.add_edge(
                    u,
                    v,
                    geometry=shape(geom),
                    **props
                )
        else:
            print("Throw not correct format")

In [20]:
import osmnx as ox
import geopandas as gpd
from shapely.geometry import LineString
from func import getNearestNodeId

# lat, long
source = [15.13022502299816, 120.58941583884963]
dest = [15.167239872374454, 120.57890970311541]


srcNode = getNearestNodeId(G, source[0], source[1])
destNode = getNearestNodeId(G, dest[0], dest[1])

routesResultsList = []

theList = None
nextG = G.copy()
for i in range(0,3):
    theList = list(ox.k_shortest_paths(nextG, srcNode, destNode, 1, weight="weight"))
    routesResultsList.append(theList[0])

    nextG = nextG.copy()

    routeToRemove = None
    for i in theList[0]:
        if G.nodes[i]['mode']['type'] != 'walk':
            routeToRemove = G.nodes[i]['mode']['details']['name']
            break

    if routeToRemove != None:
        for i in theList[0]:
            if nextG.nodes[i]['mode']['type'] != 'walk':
                if nextG.nodes[i]['mode']['details']['name'] == routeToRemove:
                    nextG.remove_node(i)

    

# Finds all permutations from the list of calculated routes differentiated by modes
permutationList = []
for i in routesResultsList:
    permutation = []
    for j in i:
        if permutation:
            modeType = G.nodes[j]['mode']['type'] if G.nodes[j]['mode']['type'] == 'walk' else G.nodes[j]['mode']['details']['name']
            if modeType != permutation[len(permutation)-1]:
                permutation.append('walk') if G.nodes[j]['mode']['type'] == 'walk' else permutation.append(G.nodes[j]['mode']['details']['name'])
        else:
            permutation.append('walk') if G.nodes[j]['mode']['type'] == 'walk' else permutation.append(G.nodes[j]['mode']['details']['name']) 


    total_weight = sum(G[u][v][0].get('weight', 0) for u, v in zip(i[:-1], i[1:]))
    # print(f"Weight: {total_weight:.2f}: {permutation}")

    if permutation not in permutationList:
        permutationList.append(permutation)

for i in permutationList:
    print(i)


['walk', 'Checkpoint-HolyAngelUniversity-Highway-Angeles', 'walk']
['walk', 'Checkpoint-HolyAngelUniversity', 'walk']
['walk', 'Marisol-Pampang', 'walk']


In [7]:
G.nodes[96]

{'osmid': 96,
 'y': 15.1335522,
 'x': 120.5918389,
 'street_count': 4,
 'highway': None,
 'ref': None,
 'mode': {'type': 'jeep',
  'details': {'name': 'Checkpoint-HolyAngelUniversity-Highway-Angeles'}}}

In [ ]:
# Find outgoing edges of a node

nodeId = 679532340884159
culprit = 1415312452801489

nodeToCheck = 2748923904

print("Outbound")
for u, v, k, data in G.out_edges(nodeToCheck, data=True, keys=True):
    print(f"{u} → {v} [key={k}] | weight: {data.get('weight')}")
print()

print("Inbound")
for u, v, k, data in G.in_edges(nodeToCheck, data=True, keys=True):
    print(f"{u} → {v} [key={k}] | weight: {data.get('weight')}")

Outbound
2748923904 → 8682825681 [key=0] | weight: 5.62
2748923904 → 8682825683 [key=0] | weight: 21.7
2748923904 → 425 [key=0] | weight: 350

Inbound
8682825680 → 2748923904 [key=0] | weight: 5.75
8682825683 → 2748923904 [key=0] | weight: 21.7
425 → 2748923904 [key=0] | weight: 10


In [ ]:
import math

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

def nearest_node(G, lat, lon):
    return min(G.nodes, key=lambda n: haversine(lat, lon, G.nodes[n]['y'], G.nodes[n]['x']))

nearest_node = nearest_node(G, 15.158640690032867, 120.59424317411658)

print(nearest_node)

13716443917


: 

In [ ]:
import itertools
with perfTimer('do shortest path'):
    pathList = ox.k_shortest_paths(G, 9174978810, 3350956127, 1, weight="weight")

print('test4')
with perfTimer('append'):
    s = list(itertools.chain.from_iterable(pathList))

print(s)

Elapsed time to do shortest path: 0.0s
test4
Elapsed time to append: 0.88s
[9174978810, 963391192211437, 89703087535231, 53344806259731, 1262400658216355, 1262400658426359, 193567543538139, 679532340884159, 1288606322887365, 1288606323622379, 54422214771407, 805902409482237, 805900650383731, 929976822189369, 69362155956295, 193567541333097, 69362155326283, 69362153331245, 29728221788595, 657944095692321, 963400898281311, 657944096112329, 657944096637339, 1415312452591485, 33064377778213, 335246335465393, 807482905781355, 752926218731157, 992992587436797, 992992587226793, 351038848789441, 351857162451071, 3350956127, 9174978810, 854298207, 89703087535231, 53344806259731, 1262400658216355, 1262400658426359, 193567543538139, 679532340884159, 1288606322887365, 1288606323622379, 54422214771407, 805902409482237, 805900650383731, 929976822189369, 69362155956295, 193567541333097, 69362155326283, 69362153331245, 29728221788595, 657944095692321, 963400898281311, 657944096112329, 657944096637339,

In [20]:
source = [15.13022502299816, 120.58941583884963]
dest = [15.167239872374454, 120.57890970311541]

from func import getNearestNodeId
srcNode = getNearestNodeId(G, source[0], source[1])
destNode = getNearestNodeId(G, dest[0], dest[1])

weightFromSource = nx.single_source_dijkstra_path_length(G, srcNode, weight='weight')
weightToDest = nx.single_source_dijkstra_path_length(G.reverse(), destNode, weight='weight')

total_est = {}

for node in G.nodes:
    if node in weightFromSource and node in weightToDest:
        total_est[node] = weightFromSource[node] + weightToDest[node]


best_estimate = min(total_est.values())
candidate_nodes = {
    n for n, cost in total_est.items()
    if cost <= best_estimate * (1 + 0.1)
}

G_sub = G.subgraph(candidate_nodes).copy()

theList = list(ox.k_shortest_paths(G_sub, srcNode, destNode, 30, weight="weight"))


# Finds all permutations from the list of calculated routes differentiated by modes
# permutationList = []
# for i in theList:
#     permutation = []
#     for j in i:
#         if permutation:
#             modeType = G.nodes[j]['mode']['type'] if G.nodes[j]['mode']['type'] == 'walk' else G.nodes[j]['mode']['details']['name']
#             if modeType != permutation[len(permutation)-1]:
#                 permutation.append('walk') if G.nodes[j]['mode']['type'] == 'walk' else permutation.append(G.nodes[j]['mode']['details']['name'])
#         else:
#             permutation.append('walk') if G.nodes[j]['mode']['type'] == 'walk' else permutation.append(G.nodes[j]['mode']['details']['name']) 


#     total_weight = sum(G[u][v][0].get('weight', 0) for u, v in zip(i[:-1], i[1:]))
#     # print(f"Weight: {total_weight:.2f}: {permutation}")

#     if permutation not in permutationList:
#         permutationList.append(permutation)

# for i in permutationList:
#     print(i)


nodes, edges = ox.graph_to_gdfs(G_sub)
nodes = nodes.reset_index(drop=True)
nodes.to_file("graph/test_graph/pruneNodes.geojson", driver='GeoJSON')
edges.to_file("graph/test_graph/pruneEdges.geojson", driver='GeoJSON')



In [ ]:
# Test k_shortest function

import osmnx as ox
from func import getNearestNodeId
import itertools

with open(f"graph_config/color.json") as f:
    colors = json.load(f)

try:
    pair = {
        'source': {'lat': float(src.split(',')[0]), 'long': float(src.split(',')[1])},
        'destination': {'lat': float(dest.split(',')[0]), 'long': float(dest.split(',')[1])},
    }

    # Gets the nearest node from each source/destination coordinates
    with perfTimer("get nearest node id"):
        srcNode = getNearestNodeId(G, pair['source']['lat'], pair['source']['long'])
        destNode = getNearestNodeId(G, pair['destination']['lat'], pair['destination']['long'])

    routesResultsList = []

    theList = None
    nextG = G.copy()
    for i in range(0,3):
        theList = list(itertools.chain.from_iterable(ox.k_shortest_paths(nextG, srcNode, destNode, 1, weight="weight")))
        routesResultsList.append(theList[0])

        nextG = nextG.copy()

        routeToRemove = None
        for i in theList[0]:
            if G.nodes[i]['mode']['type'] != 'walk':
                routeToRemove = G.nodes[i]['mode']['details']['name']
                break

        if routeToRemove != None:
            for i in theList[0]:
                if nextG.nodes[i]['mode']['type'] != 'walk':
                    if nextG.nodes[i]['mode']['details']['name'] == routeToRemove:
                        nextG.remove_node(i)
    
    resultsObj = {}
    resultsJson = {'routes': resultsObj}

    count = 1
    for i in routesResultsList:
        nodeList = []
        for j in i:
            node = G.nodes[j]
            if nodeList:
                # print(f'{node['mode']['type']}: {i}')
                if node['mode']['type'] == nodeList[len(nodeList)-1]['mode']['type']:
                    nodeList[len(nodeList)-1]['geometry'].append([float(node['x']), float(node['y'])])
                else:
                    nodeList.append({
                        'mode': {
                            'type': node['mode']['type'],
                            'details': {
                                'name': node['mode']['details']['name'] if node['mode']['type'] != 'walk' else 'On foot',
                                'color': colors[node['mode']['details']['name']] if node['mode']['type'] != 'walk' else "#005eff"
                            }
                        },
                        'geometry': [
                            [float(node['x']), float(node['y'])]
                        ] 
                    })
            else:
                nodeList.append({
                    'mode': {
                        'type': node['mode']['type'],
                        'details': {
                            'name': node['mode']['details']['name'] if node['mode']['type'] != 'walk' else 'On foot',
                            'color': colors[node['mode']['details']['name']] if node['mode']['type'] != 'walk' else "#005eff"
                        }
                    },
                    'geometry': [
                        [float(node['x']), float(node['y'])]
                    ]
                })
            resultsObj[f'result-{count}'] = nodeList
            count+=1
    return resultsJson
        
        
        

except Exception as e:
    raise Exception(status_code=500, detail=str(e))